In [24]:
! pip uninstall typing-extensions --yes
! pip install typing-extensions==4.7.1 

Found existing installation: typing_extensions 4.15.0
Uninstalling typing_extensions-4.15.0:
  Successfully uninstalled typing_extensions-4.15.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
selenium 4.36.0 requires typing_extensions<5.0,>=4.14.0, but you have typing-extensions 4.7.1 which is incompatible.


In [16]:
import pandas as pd
import numpy as np
from datetime import datetime,timedelta
from lxml import html

import requests
from bs4 import BeautifulSoup
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
#!pip install requests_html
#from requests_html import HTMLSession
import random
import re
import time 

import string
import matplotlib as mlt
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.preprocessing import LabelEncoder

import pymysql
pymysql.install_as_MySQLdb()
import MySQLdb

#! pip install wordcloud
#from subprocess import check_output
#from wordcloud import WordCloud, STOPWORDS

I ran into issues becacuse instead of having paginated data the data was on an infinite scroll. To address this I 
implemented Selenium but ran into issues with the scrape as a user auth was needed. I went around this with an automated
escape action but the issue persisted as I wasnt getting any scroll. I was able to accheive a scroll by increasing scroll size 
and wait time to allow buffer

In [21]:
def extract():
    url = 'https://www.linkedin.com/jobs/search/?f_TPR=r604800&geoId=103644278&keywords=Sport&location=United%20States '
    driver = webdriver.Chrome()
    driver.maximize_window()
    
    driver.get(url)
    
    actions = ActionChains(driver)
    actions.send_keys(Keys.ESCAPE).perform()

    last_height = 0
    unchanged_count = 0

    while True:
        # Scroll to the bottom
        driver.execute_script("window.scrollBy(0,2500)")
        time.sleep(5)  # Wait for new jobs to load

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(str(new_height)+"-"+str(last_height))

        if new_height == last_height:
            unchanged_count += 1
            print(f"No new content ({unchanged_count}/3)")
            if unchanged_count >= 3:
                print("Reached end of page.")
                break
        else:
            unchanged_count = 0
            last_height = new_height

    # Once fully scrolled, hand the full HTML to BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    return soup

In [39]:
def merge(dict1, dict2):
    return(dict2.update(dict1))

def extract_inner(url_inner):
       
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"]
    
    headers = {"accept":"*/*",
               "accept-encoding": "gzip, deflate, br, zstd",
               "accept-language": "en-US,en;q=0.9",
               'User-Agent': random.choice(user_agents_list)}

    r_inner = requests.get(url_inner,headers)
    soup_inner = BeautifulSoup(r_inner.content,'html.parser')
    return(soup_inner)

def transform(soup):
    divs = soup.find_all('div',class_ = 'base-card relative w-full hover:no-underline focus:no-underline base-card--link base-search-card base-search-card--link job-search-card')
    
    for job in divs:
        raw_job={         
                'title': job.find('h3',class_='base-search-card__title').get_text.strip() if job.find('h3',class_='base-search-card__title') else None,
                'company': job.find('h4',class_ = 'base-search-card__subtitle').get_text.strip() if job.find('h4',class_ = 'base-search-card__subtitle') else None,
                'location_info': job.find('span',class_ = 'job-search-card__location').get_text.strip() if job.find('span',class_ = 'job-search-card__location') else None,
                }
        
        raw_job['job_url'] = job.a['href'].partition('?position')[0]
        raw_job['job_id'] = raw_job['job_url'].split('-')[-1])
        raw_job['scrape_datetime'] = datetime.now()         
        
        #details = []
        more_info = extract_inner(link_ext)
        more_info.find('span', 'posted-time-ago__text topcard__flavor--metadata').get_text().strip()
        
        try:
            divs_inner = more_info.find('div',class_ = 'show-more-less-html__markup')
            divs_inner_1 = divs_inner.find_all('ul')
            base_text = divs_inner.prettify()
            details = []
        
            for info in divs_inner_1:
                for i in (info.find_all('li')):
                    details.append(i.text.strip())
            
        except:
            details= []
        
        job = {
            'job_ID': job_ID,
            'title': title,
            'Location': location,
            'Company': company,
            'details': details,
            'url': link_ext,
            'posting_datetime': post_date.strftime("%m/%d/%Y %H:%M:%S"),
            'scrape_datetime': scrape_date.strftime("%m/%d/%Y %H:%M:%S"),
            'additionals': base_text
        }
        
        joblist.append(job)

    return


In [10]:
joblist = []
c = extract()
transform(c)

TypeError: extract() takes 0 positional arguments but 1 was given

In [37]:
#### START HERE BECAUSE NEED TO BETTER SCRAPE INFO FROM INDIVIDUAL SITES
results = c.find('ul', class_ = "jobs-search__results-list")
jobs = results.find_all('li')
for job in jobs:
    print(job.find('h3',class_='base-search-card__title').text.strip())
    print(job.find('h4',class_ = 'base-search-card__subtitle').text.strip())
    print(job.find('span',class_ = 'job-search-card__location').text.strip())
    link_ext = job.a['href'].partition('?position')[0]
    print(link_ext)
    print(link_ext.split('-')[-1])

Manager, Athlete Marketing
Red Bull
Santa Monica, CA
https://www.linkedin.com/jobs/view/manager-athlete-marketing-at-red-bull-4371481211
4371481211


1000 - Recreation Specialist
City of Fort Walton Beach
Fort Walton Beach, FL
https://www.linkedin.com/jobs/view/1000-recreation-specialist-at-city-of-fort-walton-beach-4401451818
4401451818


Partnership Activation Coordinator
Detroit City FC
Detroit, MI
https://www.linkedin.com/jobs/view/partnership-activation-coordinator-at-detroit-city-fc-4400867347
4400867347


Sport Scientist
Angel City Football Club
Thousand Oaks, CA
https://www.linkedin.com/jobs/view/sport-scientist-at-angel-city-football-club-4402060956
4402060956


PT Operations Assistant
Houston Dynamo Football Club
Houston, TX
https://www.linkedin.com/jobs/view/pt-operations-assistant-at-houston-dynamo-football-club-4400857709
4400857709


CRM Analyst
Houston Dynamo Football Club
Houston, TX
https://www.linkedin.com/jobs/view/crm-analyst-at-houston-dynamo-football-club-43995779

In [42]:
temp_text = extract_inner(link_ext)
print(temp_text)

<!DOCTYPE html>

<html lang="en">
<head>
<meta content="d_jobs_guest_details" name="pageKey"/>
<meta content="max-image-preview:large, noarchive" name="robots"/>
<meta content="max-image-preview:large, archive" name="bingbot"/>
<!-- --><!-- --> <meta content="en_US" name="locale"/>
<!-- --> <meta data-app-version="2.0.2897" data-browser-id="85fe6365-25d3-4d83-8428-398d99ca170e" data-call-tree-id="AAZP6H+N8dgmTRvfGrSd0Q==" data-dfp-member-lix-treatment="control" data-disable-jsbeacon-pagekey-suffix="false" data-dna-member-lix-treatment="enabled" data-enable-page-view-heartbeat-tracking="" data-human-member-lix-treatment="enabled" data-is-bot="true" data-is-epd-audit-event-enabled="false" data-is-feed-sponsored-tracking-kill-switch-enabled="false" data-member-id="0" data-multiproduct-name="jobs-guest-frontend" data-network-interceptor-lix-value="control" data-page-instance="urn:li:page:d_jobs_guest_details;/qikVFgSSlCEcPSQg7ma8w==" data-recaptcha-v3-integration-lix-value="control" data-s

In [50]:
print(temp_text.find('span', class_ = 'posted-time-ago__text topcard__flavor--metadata').get_text().strip())
print(temp_text.find('span', class_ = "description__job-criteria-text description__job-criteria-text--criteria").get_text().strip())
print(temp_text.find('div', class_ = "show-more-less-html__markup show-more-less-html__markup--clamp-after-5 relative overflow-hidden").get_text())

2 days ago
Mid-Senior level


In [12]:
temp_df = pd.DataFrame(joblist)

In [33]:
print(results)

<ul class="jobs-search__results-list">
<li>
<div class="base-card relative w-full hover:no-underline focus:no-underline base-card--link base-search-card base-search-card--link job-search-card" data-column="1" data-entity-urn="urn:li:jobPosting:4371481211" data-reference-id="fRwRUXBb2Z2voIOTEE6s8Q==" data-row="1" data-tracking-id="UigIxT0bpg0ehTwZIPpDog==">
<a class="base-card__full-link absolute top-0 right-0 bottom-0 left-0 p-0 z-[2] outline-offset-[4px]" data-tracking-client-ingraph="" data-tracking-control-name="public_jobs_jserp-result_search-card" data-tracking-will-navigate="" href="https://www.linkedin.com/jobs/view/manager-athlete-marketing-at-red-bull-4371481211?position=1&amp;pageNum=0&amp;refId=fRwRUXBb2Z2voIOTEE6s8Q%3D%3D&amp;trackingId=UigIxT0bpg0ehTwZIPpDog%3D%3D">
<span class="sr-only">
              
        
        Manager, Athlete Marketing
      
      
          </span>
</a>
<div class="search-entity-media">
<img alt="" aria-busy="false" class="artdeco-entity-image

In [ ]:
joblist = []
leagues = ['Major%20League%20Soccer', 'Major%20League%20Baseball','National%20Football%20League']
for league in leagues:
    c=extract(league)
    transform(c)
    
joblist1 = joblist

In [ ]:
joblist = []
leagues_again = ['National%20Hockey%20League', 'National%20Basketball%20Association']
for league in leagues_again:
    c=extract(league)
    transform(c)
    
joblist2 = joblist

In [ ]:
database = MySQLdb.connect(host="localhost" , user="root" , passwd="Pps11844")
cursor = database.cursor()

def execute_query(query_statement):
    try:
        cursor.execute(query_statement);
        database.commit();
        print("Data is Succefully Inserted")
    
    except Exception as e :
        database.rollback();
        print("The  Exception Occured : ", e)

execute_query("USE JobsinSports")

SQL_df_posting = pd.read_sql('select * from job_posting',database)

SQL_df_companies = pd.read_sql('select * from company_team',database)

cursor.execute("SELECT MAX(company_ID) FROM company_team;")
result = cursor.fetchone();
max_comp_ID = result[0]

database.close()

In [ ]:
job_posting_linkedin_1 = pd.DataFrame(joblist1)
job_posting_linkedin_2 = pd.DataFrame(joblist2)
job_posting_linkedin = pd.concat([job_posting_linkedin_1, job_posting_linkedin_2])
job_posting_linkedin.reset_index(drop=True, inplace=True)

job_posting_linkedin['job_ID'] = job_posting_linkedin['job_ID'].astype(float)

for i,j in job_posting_linkedin.iterrows():
    if(re.findall(r'\$',j['additionals'])):
        job_posting_linkedin.at[i,'salary'] = '$'+(j['additionals'].partition('$')[2].partition('.')[0].partition('<')[0].partition('/')[0].partition('\\')[0].replace(' ','').replace('to','-').replace('TO','-').replace('K',',000').replace('k',',000').replace('\n',',000'))
    else:
        job_posting_linkedin.at[i,'salary'] = 'NA'

for i,j in job_posting_linkedin.iterrows():
    if(len(j['salary'])==3):
        job_posting_linkedin.at[i,'salary'] = (j['salary'] + '/hr')
    else:
        pass

for i,j in job_posting_linkedin.iterrows():
    if(re.findall(r'New York City',j['Location'])):
        job_posting_linkedin.at[i,'Location'] = 'New York, NY'
    else:
        pass
    
job_posting_linkedin['job_city'] = job_posting_linkedin['Location'].str.partition(",")[0]
job_posting_linkedin['job_state'] = job_posting_linkedin['Location'].str.partition(",")[2]

job_posting_linkedin['Company'] = job_posting_linkedin['Company'].str.partition('(')[0].str.replace('Football Club ','FC').str.replace('Football Club','FC').str.replace('Soccer Club','SC')
job_posting_linkedin['Company'] = job_posting_linkedin['Company'].str.strip()
job_posting_linkedin['posting_source_ID'] = 3
job_posting_linkedin['application_deadline'] = 'Unknown'
job_posting_linkedin['scrape_datetime'] = pd.to_datetime(job_posting_linkedin['scrape_datetime'])
job_posting_linkedin['posting_datetime'] = pd.to_datetime(job_posting_linkedin['posting_datetime'])


job_requirements_df = pd.DataFrame(job_posting_linkedin[['job_ID','details']])
job_requirements_df_final = job_requirements_df.assign(temp = job_requirements_df.details.str.split(",")).explode('details').drop('temp',axis=1)
job_requirements_df_final['details'] = job_requirements_df_final['details'].str.replace("'","").str.replace('"','')

In [ ]:
job_posting_linkedin.drop(['Location','details','additionals'],axis = 1,inplace = True)

In [ ]:
Company_Team = pd.DataFrame(job_posting_linkedin['Company'])
Company_Team_df = Company_Team.drop_duplicates()

In [ ]:
Company_Team_df['Company_temp'] = [1,2,3,4,5,6,7,8]
Company_Team_df.loc[Company_Team_df['Company_temp'] == 1,'company_ID'] = int(max_comp_ID + 1)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 5,'company_ID'] = int(max_comp_ID + 2)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 6,'company_ID'] = int(max_comp_ID + 3)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 7,'company_ID'] = int(max_comp_ID + 4)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 8,'company_ID'] = int(max_comp_ID + 5)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 2,'company_ID'] = 258
Company_Team_df.loc[Company_Team_df['Company_temp'] == 3,'company_ID'] = 257
Company_Team_df.loc[Company_Team_df['Company_temp'] == 4,'company_ID'] = 255
Company_Team_df.drop('Company_temp',inplace=True,axis=1)

In [ ]:
Company_Team_df

In [ ]:
job_posting_linkedin_df = pd.merge(job_posting_linkedin, Company_Team_df, left_on="Company", right_on="Company", how='left')

In [ ]:
job_posting_linkedin_df = job_posting_linkedin_df.reindex(columns = ['job_ID','title',"company_ID",'posting_source_ID','posting_datetime','scrape_datetime','application_deadline', 'salary','job_city','job_state','url'])

In [ ]:
Sources = pd.DataFrame({'source_ID': [3], 'source_name': ['Linkedin']})

In [ ]:
# Tested but not perfected yet -- IGNORE
#job_posting_linkedin_df
#count = 1

#for i,j in Company_Team_df.iterrows():
#  print(i, j['Company'])
#    if (j['Company'] in SQL_df_companies['company_name'].values):
#        j.at[i,'company_ID'] = SQL_df_companies['company_ID']
#    else:
#        Company_Team_df.at[i,'company_ID'] = max_comp_ID + count
#        count = count + 1


In [ ]:
## Initialize connection to MYSQL
database = MySQLdb.connect(host="localhost" , user="root" , passwd="Pps11844")
cursor = database.cursor()

In [ ]:
def execute_query(query_statement):
    try:
        cursor.execute(query_statement);
        database.commit();
        print("Data is Succefully Inserted")
    
    except Exception as e :
        database.rollback();
        print("The  Exception Occured : ", e)


In [ ]:
execute_query("USE JobsinSports")


In [ ]:
for i,j in job_requirements_df_final.iterrows():
    execute_query('INSERT INTO Job_Requirements (job_ID, requirements) VALUES (%d, "%s")' % (j['job_ID'],j['details']))

In [ ]:
for i,j in Sources.iterrows():
    execute_query('INSERT INTO Sources (source_ID, source_name) VALUES (%d, "%s")' % (j['source_ID'],j['source_name']))

In [ ]:
for i,j in Company_Team_df.iterrows():
    execute_query('INSERT INTO Company_Team (company_ID, company_name) VALUES (%d, "%s")' % (j['company_ID'], j['Company']))

In [ ]:
for i,j in job_posting_linkedin_df.iterrows():
    execute_query('INSERT INTO Job_Posting (job_ID, job_title, company_ID, posting_datetime, scraped_datetime, salary, job_city, job_state, posting_url, source_identifier) VALUES (%d, "%s", %d, "%s","%s", "%s", "%s", "%s", "%s", %d)' % (j['job_ID'], j['title'], j['company_ID'], j['posting_datetime'], j['scrape_datetime'], j['salary'], j['job_city'], j['job_state'], j['url'], j['posting_source_ID']))

In [ ]:
database.close()